In [2]:
import os
import pandas as pd

ROOT = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.1.2_Unit_T_Signal_CI.csv"

# ------------ Config (keyword-based, no regex) ------------

# A line is considered a "Gradle command line" if it contains ANY of these
GRADLE_INVOCATION_KEYWORDS = ["gradlew", "gradle"]

# If a Gradle command line contains ANY of these, we treat it as a unit-test signal
UNIT_TEST_TASK_KEYWORDS = [
    # explicit unit test tasks (strong)
    "testunittest",           # catches squashed names like testDebugUnitTest
    "testdebugunittest",
    "testreleaseunittest",
    "test",                   # generic 'test' task
    "check",                  # 'check' usually depends on unit tests
    "build"                   # 'build' runs unit tests unless excluded
]

# If a Gradle command line contains ANY of these, we treat it as skipping tests
EXCLUDE_KEYWORDS = [
    "-x test",
    "--exclude-task test",
    "-dskiptests",            # matches -DskipTests after lowercasing
]

# ---------------------------------------------------------

def extract_full_name(filename: str, dirpath: str) -> str:
    """
    Extract owner.repo from saved filename formats like owner.repo__Type++<path>.yml.
    Fallback: parent folder name.
    """
    base = os.path.basename(filename)
    if "__" in base:
        return base.split("__", 1)[0]
    return os.path.basename(dirpath)

def line_has_any(s: str, keywords) -> bool:
    s = s.lower()
    return any(k in s for k in keywords)

def strip_inline_yaml_comment(line: str) -> str:
    """
    Remove inline YAML comments starting at the first '#' that is NOT inside quotes.
    Preserves '#' inside single or double quoted strings.
    """
    in_single = False
    in_double = False
    i = 0
    out = []

    while i < len(line):
        ch = line[i]
        if ch == "'" and not in_double:
            # YAML single quotes escape as doubled ''
            if in_single and i + 1 < len(line) and line[i + 1] == "'":
                out.append("''")
                i += 2
                continue
            in_single = not in_single
            out.append(ch)
        elif ch == '"' and not in_single:
            # toggle double quotes; naive (good enough for CI lines)
            in_double = not in_double
            out.append(ch)
        elif ch == "#" and not in_single and not in_double:
            break  # start of comment
        else:
            out.append(ch)
        i += 1

    return "".join(out)

def detect_unit_test_signal_keywords(text: str) -> bool:
    """
    Keyword-only heuristic:
    - Ignore whole-line comments (# ...).
    - Strip inline comments (not inside quotes).
    - Look for gradle invocations that include unit-test task indicators.
    - Ignore lines that also include skip/exclude hints.
    """
    if not text:
        return False

    for raw in text.splitlines():
        if not raw.strip():
            continue
        if raw.lstrip().startswith("#"):        # whole-line comment
            continue

        code = strip_inline_yaml_comment(raw).strip()
        if not code:
            continue

        low = code.lower()

        # must look like a gradle invocation
        if not line_has_any(low, GRADLE_INVOCATION_KEYWORDS):
            continue

        # exclude skip flags
        if line_has_any(low, EXCLUDE_KEYWORDS):
            continue

        # unit-test signal on that gradle line?
        if line_has_any(low, UNIT_TEST_TASK_KEYWORDS):
            return True

    return False

# --- Scan & analyze ---
rows = []
for dirpath, _, files in os.walk(ROOT):
    for fname in files:
        if not (fname.endswith(".yml") or fname.endswith(".yaml")):
            continue

        fpath = os.path.join(dirpath, fname)
        try:
            with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read()
        except Exception:
            content = ""

        full_name = extract_full_name(fname, dirpath)
        signal = detect_unit_test_signal_keywords(content)

        rows.append({
            "filename": os.path.basename(fpath),
            "full_name": full_name,
            "unit_t_signal_ci": bool(signal),
        })

# --- Write CSV ---
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(rows)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.1.2_Unit_T_Signal_CI.csv (rows=12667)


In [3]:
import os
import re
import pandas as pd

# === FOCUS ONLY ON THIS FOLDER ===
ROOT = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"

OUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.1.2_Unit_T_Signal_Config.csv"

# ---------- helpers ----------

def is_build_gradle_file(fname: str) -> bool:
    """
    True if the filename is a Gradle build file:
    - build.gradle / build.gradle.kts
    - build__<n>.gradle / build__<n>.gradle.kts
    - or saved with '++' naming like owner.repo__Type++build__3.gradle
    """
    base = os.path.basename(fname)
    # Case 1: "owner.repo__Type++something"
    if "++" in base:
        tail = base.split("++", 1)[1]
        # normalize "__<digits>" before .gradle / .gradle.kts
        tail = re.sub(r"__\d+(?=\.gradle(?:\.kts)?$)", "", tail, flags=re.IGNORECASE)
        return tail.lower() in ("build.gradle", "build.gradle.kts")
    # Case 2: plain file in repo tree
    return bool(re.match(r"(?i)^build(?:__\d+)?\.gradle(?:\.kts)?$", base))

def extract_full_name_from_saved(fname: str, fpath: str) -> str:
    """
    Extract owner.repo from saved filenames like owner.repo__Type++<path>.
    Fallback to parent directory name if pattern not present.
    """
    base = os.path.basename(fname)
    if "__" in base:
        return base.split("__", 1)[0]
    # fallback: nearest directory name
    return os.path.basename(os.path.dirname(fpath))

def strip_comments_gradle(text: str) -> str:
    """
    Remove block comments /* ... */ and line comments // ... .
    Keeps 'http://'-style URLs by not stripping '//' preceded by ':'.
    """
    if not text:
        return ""
    # remove block comments
    s = re.sub(r"/\*.*?\*/", "", text, flags=re.DOTALL)
    # remove line comments (avoid http://)
    s = re.sub(r"(?<!:)//.*?$", "", s, flags=re.MULTILINE)
    return s

def has_unit_test_config(text: str) -> bool:
    """
    Keyword-based signals that the *build configuration* enables/uses unit tests.
    (Comments are expected to be stripped beforehand.)
    """
    if not text:
        return False

    t = text.lower()

    # Whole-file signals
    if ("testoptions" in t and "unittests" in t):   # android { testOptions { unitTests { ... } } }
        return True
    if "usejunitplatform()" in t:                   # JUnit 5
        return True
    if "org.junit.jupiter" in t or "junit:junit" in t:  # common test deps
        return True
    if ("testing" in t and "suites" in t and "test" in t):  # Gradle Test Suites plugin
        return True

    # Line-level signals (avoid matching androidTest* variants)
    for raw in t.splitlines():
        line = raw.strip()
        if not line:
            continue

        # dependency configurations for unit tests
        if "testimplementation" in line and "androidtestimplementation" not in line:
            return True
        if "testapi" in line and "androidtestapi" not in line:
            return True
        if "testcompileonly" in line and "androidtestcompileonly" not in line:
            return True
        if "testruntimeonly" in line and "androidtestruntimeonly" not in line:
            return True
        if "testcompile" in line and "androidtestcompile" not in line:
            return True
        if "testfixtures" in line:
            return True

        # tasks configuration for unit tests
        if "withtype" in line and ("<test" in line or "(test" in line):
            return True
        if ("tasks.register" in line or "tasks.named" in line) and ("('test')" in line or '("test")' in line):
            return True
        if "task test" in line:  # Groovy DSL custom test task config
            return True

        # sourcesets test config
        if "sourcesets" in line and "test" in line:
            return True

    return False

# ---------- scan & analyze ----------

rows = []
for dirpath, _, files in os.walk(ROOT):
    for fname in files:
        if not is_build_gradle_file(fname):
            continue

        fpath = os.path.join(dirpath, fname)
        try:
            with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read()
        except Exception:
            content = ""

        content_no_comments = strip_comments_gradle(content)
        signal = has_unit_test_config(content_no_comments)

        rows.append({
            "filename": os.path.basename(fpath),
            "full_name": extract_full_name_from_saved(fname, fpath),
            "unit_t_signal_config": bool(signal),
        })

# ---------- write csv ----------
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(rows)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.1.2_Unit_T_Signal_Config.csv (rows=21280)
